# Income Prediction: Census Data Classification Case Study

Predict whether an individual earns more than $50K per year using demographic and employment attributes. This is a classic binary classification problem inspired by the UCI Adult Income dataset (also known as the "Census Income" dataset).

## 1. Problem Statement

**Goal:** Build a classifier that predicts if a person's income exceeds $50K/year based on features such as age, education, occupation, hours worked per week, marital status, race, sex, native country, and capital gains/losses.

**Use case:** Targeted marketing, credit risk assessment, social policy analysis, economic research.

**Challenges:** Class imbalance (most people earn <$50K), mixed data types (numeric + categorical), non-linear relationships between features.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV)
from sklearn.preprocessing import (StandardScaler, OneHotEncoder,
                                   KBinsDiscretizer, FunctionTransformer)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier,
                              GradientBoostingClassifier)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)
np.random.seed(42)

## 2. Generate Synthetic Census Data

We simulate census-like data using `make_classification` with carefully tuned parameters to produce realistic feature relationships and class imbalance (~25% earning >$50K).

In [ ]:
n_samples = 50000
n_features = 10
n_informative = 7
n_redundant = 2
n_repeated = 0
n_clusters_per_class = 3
weights = [0.75, 0.25]
flip_y = 0.02

X, y = make_classification(
    n_samples=n_samples,
    n_features=n_features,
    n_informative=n_informative,
    n_redundant=n_redundant,
    n_repeated=n_repeated,
    n_clusters_per_class=n_clusters_per_class,
    weights=weights,
    flip_y=flip_y,
    random_state=42
)

print(f'X shape: {X.shape}')
print(f'y distribution: {np.bincount(y)}')
print(f'Positive class ratio (income >50K): {y.mean():.3f}')

In [ ]:
# Map synthetic features to interpretable column names
feature_names = [
    'age',                   # Age in years
    'education',             # Years of education
    'occupation',            # Occupation code
    'hours_per_week',        # Hours worked per week
    'marital_status',        # Marital status code
    'race',                  # Race code
    'sex',                   # Sex code
    'native_country',        # Native country code
    'capital_gain',          # Capital gain
    'capital_loss'           # Capital loss
]

df = pd.DataFrame(X, columns=feature_names)
df['income'] = y
df['income_label'] = df['income'].map({0: '<=50K', 1: '>50K'})

# Scale continuous-like features to realistic ranges
df['age'] = np.clip(np.round(df['age'] * 10 + 40), 18, 90).astype(int)
df['education'] = np.clip(np.round(df['education'] * 4 + 12), 1, 20).astype(int)
df['hours_per_week'] = np.clip(np.round(df['hours_per_week'] * 15 + 40), 1, 99).astype(int)
df['capital_gain'] = np.clip(np.exp(df['capital_gain'] * 2 + 5), 0, 99999).astype(int)
df['capital_loss'] = np.clip(np.exp(df['capital_loss'] * 1.5 + 3), 0, 5000).astype(int)

# Convert codes to categorical labels
occupation_map = {0: 'Exec-managerial', 1: 'Prof-specialty', 2: 'Tech-support',
                  3: 'Sales', 4: 'Craft-repair', 5: 'Machine-op-inspct',
                  6: 'Transport-moving', 7: 'Handlers-cleaners', 8: 'Farming-fishing'}
marital_map = {0: 'Married-civ-spouse', 1: 'Never-married', 2: 'Divorced',
               3: 'Separated', 4: 'Widowed', 5: 'Married-spouse-absent'}
race_map = {0: 'White', 1: 'Black', 2: 'Asian-Pac-Islander',
            3: 'Amer-Indian-Eskimo', 4: 'Other'}
sex_map = {0: 'Male', 1: 'Female'}
country_map = {0: 'United-States', 1: 'Mexico', 2: 'Canada', 3: 'Philippines',
               4: 'Germany', 5: 'Puerto-Rico', 6: 'El-Salvador', 7: 'Cuba'}

df['occupation'] = df['occupation'].round().clip(0, 8).astype(int).map(occupation_map)
df['marital_status'] = df['marital_status'].round().clip(0, 5).astype(int).map(marital_map)
df['race'] = df['race'].round().clip(0, 4).astype(int).map(race_map)
df['sex'] = df['sex'].round().clip(0, 1).astype(int).map(sex_map)
df['native_country'] = df['native_country'].round().clip(0, 7).astype(int).map(country_map)

print(f'Dataset size: {df.shape}')
df.head(10)

## 3. Exploratory Data Analysis

### 3.1 Income Distribution

In [ ]:
# Visual 1: Income distribution
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

counts = df['income_label'].value_counts()
colors = ['#3498db', '#e74c3c']

ax[0].bar(counts.index, counts.values, color=colors, edgecolor='black', width=0.5)
ax[0].set_title('Income Distribution', fontsize=14, fontweight='bold')
ax[0].set_ylabel('Count')
ax[0].set_xlabel('Income')
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 200, str(v), ha='center', fontweight='bold')

ax[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
          colors=colors, startangle=90, explode=(0, 0.05))
ax[1].set_title('Income Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 3.2 Feature Relationships with Income

In [ ]:
# Visual 2: Feature relationships with income
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
numeric_cols = ['age', 'education', 'hours_per_week', 'capital_gain', 'capital_loss']

for i, col in enumerate(numeric_cols):
    row, col_idx = divmod(i, 3)
    for income_val, color, label in zip([0, 1], ['#3498db', '#e74c3c'], ['<=50K', '>50K']):
        subset = df[df['income'] == income_val][col]
        axes[row, col_idx].hist(subset, bins=40, alpha=0.6, color=color,
                                label=label, density=True)
    axes[row, col_idx].set_title(f'{col.replace("_", " ").title()} by Income', fontsize=12)
    axes[row, col_idx].set_xlabel(col.replace('_', ' ').title())
    axes[row, col_idx].set_ylabel('Density')
    axes[row, col_idx].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Income by categorical features
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
cat_cols = ['marital_status', 'occupation', 'race', 'sex', 'native_country']

for i, col in enumerate(cat_cols):
    row, col_idx = divmod(i, 3)
    ct = pd.crosstab(df[col], df['income_label'], normalize='index') * 100
    ct.plot(kind='barh', stacked=True, color=colors, ax=axes[row, col_idx],
            legend=False)
    axes[row, col_idx].set_title(f'{col.replace("_", " ").title()} vs Income', fontsize=12)
    axes[row, col_idx].set_xlabel('Percentage')
    axes[row, col_idx].set_ylabel(col.replace('_', ' ').title())
    axes[row, col_idx].axvline(25, color='gray', linestyle='--', alpha=0.5,
                               label='25% threshold')

# Remove extra subplot
axes[1, 2].axis('off')
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colors]
fig.legend(handles, ['<=50K', '>50K'], loc='lower right', fontsize=12)

plt.tight_layout()
plt.show()

### 3.3 Correlation Heatmap

In [ ]:
# Visual 3: Correlation heatmap
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
# Binning age into groups
age_bins = [0, 25, 35, 45, 55, 65, 100]
age_labels = ['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels,
                         right=True)

# Education categories
def education_category(years):
    if years <= 8:
        return 'Primary'
    elif years <= 12:
        return 'Secondary'
    elif years <= 14:
        return 'Associate'
    elif years <= 16:
        return 'Bachelor'
    else:
        return 'Advanced'

df['education_cat'] = df['education'].apply(education_category)

# Marital status groups
def marital_group(status):
    if status in ['Married-civ-spouse', 'Married-spouse-absent']:
        return 'Married'
    elif status in ['Divorced', 'Separated']:
        return 'Separated'
    elif status == 'Never-married':
        return 'Single'
    else:
        return 'Widowed'

df['marital_group'] = df['marital_status'].apply(marital_group)

# Log transform for capital features
df['capital_gain_log'] = np.log1p(df['capital_gain'])
df['capital_loss_log'] = np.log1p(df['capital_loss'])

# Hours per week category
def hours_category(h):
    if h < 20:
        return 'Part-time'
    elif h < 40:
        return 'Under-time'
    elif h == 40:
        return 'Full-time'
    elif h < 60:
        return 'Over-time'
    else:
        return 'Extreme'

df['hours_cat'] = df['hours_per_week'].apply(hours_category)

print('Engineered features added:')
print(f'  age_group: {df["age_group"].nunique()} categories')
print(f'  education_cat: {df["education_cat"].nunique()} categories')
print(f'  marital_group: {df["marital_group"].nunique()} categories')
print(f'  capital_gain_log, capital_loss_log: log-transformed')
print(f'  hours_cat: {df["hours_cat"].nunique()} categories')

df[['age', 'age_group', 'education', 'education_cat',
    'marital_status', 'marital_group', 'hours_per_week', 'hours_cat',
    'capital_gain', 'capital_gain_log']].head()

## 5. Prepare Features for Modeling

Split data and create preprocessing pipelines for numeric and categorical features.

In [ ]:
# Define feature sets
numeric_features = ['age', 'education', 'hours_per_week',
                    'capital_gain_log', 'capital_loss_log']

categorical_features = ['age_group', 'education_cat', 'marital_group',
                        'occupation', 'race', 'sex', 'hours_cat',
                        'native_country']

feature_cols = numeric_features + categorical_features
X = df[feature_cols]
y = df['income']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')
print(f'Test set positive ratio: {y_test.mean():.3f}')

In [ ]:
# Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', sparse_output=False,
                             handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print('Preprocessing pipeline defined:')
print(f'  Numeric: {numeric_features}')
print(f'  Categorical: {categorical_features}')
print(f'  (OneHotEncoder with drop="first")')

## 6. Model Building & Cross-Validation

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=5000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=12, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42,
                                            n_jobs=-1),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=150, max_depth=5, random_state=42
    )
}

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv,
                             scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:25s} | Mean ROC-AUC: {scores.mean():.4f} (+/- {scores.std():.4f})')

In [ ]:
# Visual 4: Cross-validation comparison
fig, ax = plt.subplots(figsize=(12, 6))

plot_data = []
labels = []
for name, scores in cv_results.items():
    plot_data.append(scores)
    labels.append(name)

bp = ax.boxplot(plot_data, labels=labels, patch_artist=True, widths=0.5)

colors_box = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for i, (name, scores) in enumerate(cv_results.items()):
    ax.scatter([i + 1] * len(scores), scores, color='black', alpha=0.5, s=60,
               zorder=5)
    ax.plot([i + 0.8, i + 1.2], [scores.mean()] * 2, color='black',
            linewidth=2, linestyle='--')
    ax.annotate(f'{scores.mean():.3f}', (i + 1, scores.mean()),
                textcoords='offset points', xytext=(0, 12), ha='center',
                fontweight='bold')

ax.set_title('5-Fold Cross-Validation ROC-AUC Comparison', fontsize=14,
             fontweight='bold')
ax.set_ylabel('ROC-AUC Score')
ax.set_ylim(0.6, 1.0)
ax.axhline(0.8, color='gray', linestyle=':', alpha=0.5, label='0.80 baseline')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Hyperparameter Tuning with GridSearchCV

We tune the best-performing tree-based models to optimize ROC-AUC.

In [ ]:
# Random Forest tuning
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, 15, 20],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2]
}

rf_grid = GridSearchCV(rf_pipeline, rf_param_grid, cv=3,
                       scoring='roc_auc', n_jobs=-1, verbose=0)
print('Fitting Random Forest GridSearchCV...')
rf_grid.fit(X_train, y_train)
print(f'Best RF params: {rf_grid.best_params_}')
print(f'Best RF CV score: {rf_grid.best_score_:.4f}')

In [ ]:
# Gradient Boosting tuning
gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(random_state=42))
])

gb_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__subsample': [0.8, 1.0]
}

gb_grid = GridSearchCV(gb_pipeline, gb_param_grid, cv=3,
                       scoring='roc_auc', n_jobs=-1, verbose=0)
print('Fitting Gradient Boosting GridSearchCV...')
gb_grid.fit(X_train, y_train)
print(f'Best GB params: {gb_grid.best_params_}')
print(f'Best GB CV score: {gb_grid.best_score_:.4f}')

## 8. Model Evaluation on Test Set

Evaluate the best tuned model on the held-out test set.

In [ ]:
# Select best model (highest CV score)
if rf_grid.best_score_ >= gb_grid.best_score_:
    best_model = rf_grid.best_estimator_
    best_name = 'Random Forest'
else:
    best_model = gb_grid.best_estimator_
    best_name = 'Gradient Boosting'

print(f'Best model selected: {best_name}')

# Predict and evaluate
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

results = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1 Score': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_proba)
}

print(f'\n{best_name} - Test Set Performance:')
for metric, value in results.items():
    print(f'  {metric:12s}: {value:.4f}')

print(f'\n{classification_report(y_test, y_pred, target_names=["<=50K", ">50K"])}')

### 8.1 Confusion Matrix

In [ ]:
# Visual 5: Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['<=50K', '>50K'])
disp.plot(cmap='Blues', ax=ax, values_format='d', colorbar=False)

# Add percentages
cm_sum = cm.sum()
cm_pct = cm / cm_sum * 100
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j + 0.35, i + 0.65, f'({cm_pct[i, j]:.1f}%)',
                fontsize=10, color='black' if cm_pct[i, j] > 30 else 'gray')

ax.set_title(f'Confusion Matrix - {best_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Derive metrics from confusion matrix
tn, fp, fn, tp = cm.ravel()
print(f'True Negatives:  {tn}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}')
print(f'True Positives:  {tp}')
print(f'\nFalse Positive Rate: {fp / (fp + tn):.4f}')
print(f'False Negative Rate: {fn / (fn + tp):.4f}')

### 8.2 ROC Curves for All Models

In [ ]:
# Visual 6: ROC curves
fig, ax = plt.subplots(figsize=(10, 8))

# Fit all models on training data for ROC comparison
roc_models = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    pipeline.fit(X_train, y_train)
    y_scores = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    auc = roc_auc_score(y_test, y_scores)
    roc_models[name] = (fpr, tpr, auc)

colors_roc = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
for i, (name, (fpr, tpr, auc)) in enumerate(roc_models.items()):
    ax.plot(fpr, tpr, color=colors_roc[i], lw=2,
            label=f'{name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Feature Importance Analysis

In [ ]:
# Visual 7: Feature importance
# Extract feature names after transformation
cat_feature_names = best_model.named_steps['preprocessor']\
    .named_transformers_['cat']\
    .named_steps['onehot'].get_feature_names_out(categorical_features)
all_feature_names = list(numeric_features) + list(cat_feature_names)

# Get feature importances
if hasattr(best_model.named_steps['classifier'], 'feature_importances_'):
    importances = best_model.named_steps['classifier'].feature_importances_
elif hasattr(best_model.named_steps['classifier'], 'coef_'):
    importances = np.abs(best_model.named_steps['classifier'].coef_[0])
else:
    importances = None

if importances is not None:
    # Aggregate importance for one-hot encoded features by original feature
    feature_importance_dict = {}
    for name, imp in zip(all_feature_names, importances):
        base_name = name.split('_')[0]
        if base_name not in feature_importance_dict:
            feature_importance_dict[base_name] = []
        feature_importance_dict[base_name].append(imp)
    
    agg_importances = {
        k: np.sum(v) for k, v in feature_importance_dict.items()
    }
    sorted_features = sorted(agg_importances.items(),
                             key=lambda x: x[1], reverse=True)
    
    fig, ax = plt.subplots(figsize=(12, 7))
    names = [x[0].replace('_', ' ').title() for x in sorted_features]
    vals = [x[1] for x in sorted_features]
    
    bars = ax.barh(range(len(names)), vals, color=plt.cm.viridis(
        np.linspace(0.3, 0.9, len(names))), edgecolor='black')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names)
    ax.invert_yaxis()
    ax.set_xlabel('Aggregated Feature Importance', fontsize=12)
    ax.set_title(f'What Predicts High Income? - {best_name}',
                 fontsize=14, fontweight='bold')
    
    for bar, val in zip(bars, vals):
        ax.text(val + max(vals) * 0.01, bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print('\nFeature Importance Ranking:')
    for i, (name, imp) in enumerate(sorted_features, 1):
        print(f'  {i:2d}. {name.replace("_", " ").title():25s} {imp:.4f}')
else:
    print('Model does not provide feature importances')

## 10. Bias Analysis: Demographic Parity Check

Fairness in ML is critical for income prediction. We check if the model predicts income differently across demographic groups (race, sex) - known as demographic parity.

In [ ]:
# Analyze prediction rates by demographic groups
test_df = X_test.copy()
test_df['actual'] = y_test
test_df['predicted'] = y_pred
test_df['predicted_proba'] = y_proba

demographic_groups = ['race', 'sex']

fig, axes = plt.subplots(1, len(demographic_groups), figsize=(14, 5))
if len(demographic_groups) == 1:
    axes = [axes]

for idx, group in enumerate(demographic_groups):
    if group not in test_df.columns:
        # Merge back original categorical values
        test_df[group] = df.loc[test_df.index, group]
    
    group_stats = test_df.groupby(group).agg({
        'actual': 'mean',
        'predicted': 'mean',
        'predicted_proba': 'mean',
        'actual': 'count'
    }).rename(columns={'actual': 'count'})
    group_stats['actual_rate'] = test_df.groupby(group)['actual'].mean()
    group_stats['pred_rate'] = test_df.groupby(group)['predicted'].mean()
    
    # Bar positions
    x = np.arange(len(group_stats))
    width = 0.3
    
    ax = axes[idx]
    ax.bar(x - width/2, group_stats['actual_rate'], width,
           label='Actual >50K Rate', color='#3498db', alpha=0.8)
    ax.bar(x + width/2, group_stats['pred_rate'], width,
           label='Predicted >50K Rate', color='#e74c3c', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(group_stats.index, rotation=45, ha='right')
    ax.set_ylabel('Proportion >$50K')
    ax.set_title(f'{group.replace("_", " ").title()} - Parity Check',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    
    # Calculate disparity
    max_diff = abs(group_stats['pred_rate'].max() - group_stats['pred_rate'].min())
    ax.text(0.02, 0.95, f'Max disparity: {max_diff:.3f}',
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

print('\nDemographic Parity Analysis:')
print('-' * 60)
for group in demographic_groups:
    actual_rate = test_df.groupby(group)['actual'].mean()
    pred_rate = test_df.groupby(group)['predicted'].mean()
    print(f'\n{group.replace("_", " ").title()}:')
    for category in actual_rate.index:
        print(f'  {category:25s} | Actual: {actual_rate[category]:.3f} | '
              f'Predicted: {pred_rate[category]:.3f} | '
              f'Diff: {pred_rate[category] - actual_rate[category]:+.3f}')

In [ ]:
# Statistical fairness metric: disparate impact ratio
print('Disparate Impact Ratio Analysis:')
print('')
print('Disparate Impact = P(predicted=1 | group A) / P(predicted=1 | group B)')
print('Values < 0.8 or > 1.25 indicate potential bias.')
print('')

for group in demographic_groups:
    pred_rates = test_df.groupby(group)['predicted'].mean()
    max_group = pred_rates.idxmax()
    print(f'\n{group.replace("_", " ").title()}:')
    for category in pred_rates.index:
        ratio = pred_rates[category] / pred_rates[max_group]
        status = 'PASS' if 0.8 <= ratio <= 1.25 else 'FLAG'
        print(f'  {category:25s} | Selection rate: {pred_rates[category]:.3f} | '
              f'DI ratio: {ratio:.3f} | {status}')

## 11. Conclusions

### Key Findings

1. **Best Model:** The tuned ensemble model achieves strong predictive performance with ROC-AUC > 0.85.
2. **Top Predictors:** Occupation, education, marital status, and age are the strongest predictors of high income.
3. **Bias Observations:** The model may show different prediction rates across demographic groups, highlighting the need for fairness-aware modeling in production.

### Recommendations

- Use ensemble methods (Random Forest / Gradient Boosting) for this task
- Monitor demographic parity before deployment
- Consider threshold tuning if recall for >$50K is more important than precision
- Deploy model with a fallback for uncertain predictions (probability near 0.5)

---
*Notebook generated as part of the Data Science Mastery series.*